# EuroCropsML — Exploratory Data Analysis

Repository: https://github.com/dida-do/eurocropsml · Paper: [Reyes et al., *Sci. Data* 2025](https://www.nature.com/articles/s41597-025-04952-7) · Zenodo: https://zenodo.org/doi/10.5281/zenodo.10629609

**Dataset in one sentence.** 706,683 Sentinel-2 L1C per-parcel median time series across Estonia, Latvia and Portugal (year 2021), with EuroCrops HCAT crop-class labels, designed for **few-shot / transnational transfer** benchmarking.

**What this notebook does.** Code cells reproduce every figure end-to-end (`_run_analysis.py` and `_estonia_phenology.py` are the same logic, run as scripts for speed on the full disk scan). Markdown cells embed the saved figures and report the actual findings from a **30,000-parcel random sample** (~4.2% of the dataset) plus an **Estonia-only deep-dive** that scans all 175,906 Estonia parcels for the top-5-class phenology comparison.

**Verified on-disk format** (`eurocropsml` ≥ 0.4):
- One `.npz` per parcel, flat in `<DATA_ROOT>/preprocess/`, named `<NUTS3>_<parcel_id>_<HCAT_id>.npz`.
- npz keys: `data` (`(T, 13)` int64 reflectance), `dates` (`(T,)` `datetime64[D]`), `center` (`(2,)` float64, **order is `(lon, lat)`**, decimal degrees / WGS-84).
- 13 S2 bands in **fixed order**: B01, B02, B03, B04, B05, B06, B07, B08, **B8A**, B09, B10, B11, B12. NDVI uses B04 (idx 3) red and B08 (idx 7) NIR.
- Cloud filtering already applied (B04-based threshold ≤ 0.5 probability).


## 0. Setup

In [ ]:
# %pip install eurocropsml pandas numpy matplotlib seaborn tqdm pyarrow

In [ ]:
from __future__ import annotations
import os, re, json, random
from pathlib import Path

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(context="notebook", style="whitegrid")
random.seed(42); np.random.seed(42)

DATA_ROOT = Path(os.environ.get("EUROCROPSML_DATA", Path.home() / "eurocropsml_data")).expanduser()
PREPROCESS_DIR = DATA_ROOT / "preprocess"
FIG = Path("figures")
print("DATA_ROOT =", DATA_ROOT, "·", "exists:", PREPROCESS_DIR.exists())

## 1. Download

Two routes — both land at `<DATA_ROOT>/preprocess/*.npz`.

**Direct Zenodo (recommended, ~1.5 GB):** grab only the already-cloud-filtered preprocess archive.

```bash
mkdir -p ~/eurocropsml_data && cd ~/eurocropsml_data
curl -L -o preprocess.zip "https://zenodo.org/api/records/15095445/files/preprocess.zip/content"
unzip -q preprocess.zip   # creates ./preprocess/ with 706,683 .npz files
```

**Official CLI** (also fetches raw_data, ~5 GB total):

```bash
eurocropsml-cli datasets eurocrops download
```

Either way, `DATA_ROOT/preprocess` should end up holding 706,683 `.npz` files (flat, no subfolders).

## 2. On-disk inspection

Open one file to confirm the schema before doing anything else.

In [ ]:
all_npz = sorted(PREPROCESS_DIR.glob("*.npz"))
print(f"{len(all_npz):,} .npz files")
with np.load(all_npz[0]) as z:
    for k in z.files:
        a = z[k]
        print(f"  {k:8s} shape={a.shape} dtype={a.dtype} sample={a.ravel()[:3]}")

**Output observed.**

```
706,683 .npz files
  data     shape=(39, 13) dtype=int64 sample=[1328 1007  733]
  dates    shape=(39,) dtype=datetime64[D] sample=['2021-04-07' '2021-04-17' '2021-04-19']
  center   shape=(2,) dtype=float64 sample=[25.367 59.090]
```

Confirms: 13 spectral bands, int64 reflectance (no scaling applied — values are raw S2 L1C × 10 000), per-day timestamps, and a `(lon, lat)` centroid in WGS-84 degrees. Filename `EE001_19990038_3302000000.npz` decodes to NUTS-3 region `EE001`, parcel `19990038`, HCAT class `3302000000` (= "Grasslands").

## 3. Catalogue: parse filenames + cheap metadata

For exploration we don't need the band matrices — only `(country, NUTS3, parcel_id, HCAT, n_timesteps, lat, lon)`. The cell below scans **30,000 random parcels** (≈4.2% sample) so it finishes in ~6 s on a laptop. Bump `SAMPLE = None` to scan all 706k (~3 min).

In [ ]:
NUTS_PREFIX = {"EE": "Estonia", "LV": "Latvia", "PT": "Portugal"}
FNAME_RE = re.compile(r"^(?P<nuts>[A-Z]{2}[A-Z0-9]+)_(?P<pid>\d+)_(?P<cls>\d+)\.npz$")

SAMPLE = 30_000
files = random.sample(all_npz, SAMPLE) if SAMPLE and len(all_npz) > SAMPLE else all_npz

rows = []
for p in tqdm(files, desc="catalogue"):
    m = FNAME_RE.match(p.name)
    if not m: continue
    nuts = m.group("nuts")
    with np.load(p) as z:
        n_t = int(z["dates"].shape[0])
        lon, lat = float(z["center"][0]), float(z["center"][1])
    rows.append(dict(path=str(p), country=NUTS_PREFIX.get(nuts[:2], "?"), nuts=nuts,
                      parcel_id=m.group("pid"), hcat=m.group("cls"),
                      n_timesteps=n_t, lat=lat, lon=lon))
df = pd.DataFrame(rows)
df.to_parquet("catalogue_parcels.parquet", index=False)
df.head(3)

## 4. Part 1 — general understanding

All figures below were produced by running [_run_analysis.py](_run_analysis.py) on the 30 000-parcel sample. Each subsection states the **finding** and explains **what it means for the thesis**.

### 4.1 Parcels per country

![Parcels per country](figures/01_parcels_per_country.png)

**Numbers (sample of 30 000).**

| Country | Parcels | Share |
|---|---:|---:|
| Latvia   | 18 205 | 60.7 % |
| Estonia  |  7 582 | 25.3 % |
| Portugal |  4 213 | 14.0 % |

Extrapolating to the full 706 683-parcel population (and cross-checked against the actual full-disk Estonia count of **175 906** parcels), the country split is approximately **Latvia ≈ 430 k · Estonia ≈ 180 k · Portugal ≈ 100 k**. Latvia dominates by ~2.4× and Portugal is the minority — this is exactly the imbalance the paper's *transnational* protocol is designed to stress-test (small target country, large source).

**Implication.** For the few-shot probe experiment, Portugal will be the most constrained target. Sampling K-shot subsets from Portugal will be the hardest setting and the most informative for the "how few labels suffice?" research question.

### 4.2 Class distribution

![Top-30 HCAT classes](figures/02_top30_classes.png)

**Top-10 (sample of 30 000).**

| HCAT id | Parcels | Likely crop |
|---|---:|---|
| 3302000000 | 13 425 | **Grassland / pasture** |
| 3301010101 |  2 600 | Spring soft wheat |
| 3301010500 |  1 262 | Oats |
| 3399000000 |  1 227 | Other arable crops (Portugal placeholder) |
| 3301010102 |  1 192 | Winter soft wheat |
| 3301010402 |  1 069 | Barley (winter) |
| 3301110000 |    926 | Legumes |
| 3301990000 |    912 | Other cereals |
| 3301060401 |    800 | Rapeseed |
| 3303050000 |    759 | Olive groves (Portugal) |

*HCAT codes follow the EuroCrops hierarchy; first 4 digits are the level-2 group. 33-02 = grassland, 33-01 = arable, 33-03 = permanent crops.*

The distribution is **extreme long-tail**: `3302000000` (grassland) alone holds **45 %** of all parcels. 125 distinct classes appeared in the 30 k sample (paper reports 176 in the full dataset — the tail keeps adding rare classes as you scale up).

![Class coverage Lorenz curve](figures/04_lorenz_classes.png)

**Class coverage.** The Lorenz curve quantifies how few classes carry the mass:
- **Top-2** classes cover **50 %** of parcels.
- **Top-10** cover **80 %**.
- **Top-26** cover **95 %**.
- The remaining ~150 classes share the last 5 %.

**Implication.** Any classifier trained on this data will look great on macro-accuracy by predicting grassland — meaningful evaluation requires **per-class F1** and **balanced accuracy**, restricted to a curated subset of agronomically distinct classes. The paper's k-shot protocol explicitly samples balanced support sets to dodge this trap; we'll need to do the same.

### 4.3 Per-country class profile

![Top-10 per country](figures/03_top10_per_country.png)

Each country has a **distinct dominant crop mix**:
- **Estonia / Latvia** — grassland → spring wheat → cereals/oilseeds (oats, barley, rapeseed). Classic Baltic temperate-mixed farming.
- **Portugal** — `3399000000` (Portugal-specific "other arable" placeholder) → olives → grassland → vineyards (`3303030500`). Mediterranean perennial / silvo-pastoral.

### Transfer-feasibility heatmap

![Top-20 classes × country](figures/05_class_country_heatmap.png)

Of the 20 most populous classes, **only 11 are present in all three countries** — and even those 11 are heavily skewed (e.g. grassland: 9 104 LV vs 697 PT). Several Estonian/Latvian top classes have **zero parcels** in Portugal (`3301010101`, `3301010102`, `3301010402`, `3301060401`), and Portugal's top class (`3399000000`) is essentially unique to Portugal.

**Implication for transnational transfer.** Cross-country generalisation is **not** an apples-to-apples test of the same class set; it's mostly a test on a small set of **shared classes** with very different prior frequencies. When we report "EE→PT transfer accuracy," we must restrict to the intersected label space — otherwise we're reporting performance on classes that never appeared at source or target.

### 4.4 NUTS-3 region coverage

| Country | NUTS-3 regions | Top region (parcels in sample) |
|---|---:|---|
| Estonia  | 5  | EE008 (n=3 380), EE004 (n=1 887) |
| Latvia   | 5  | LV005 (n=6 238), LV008 (n=3 859) |
| Portugal | 21 | PT11E (n=993), PT16E (n=375)   |

![Portugal NUTS-3](figures/06_nuts_portugal.png)

Estonia and Latvia are split into just 5 NUTS-3 regions each (the Baltics are small), while Portugal is sliced into **21 finer regions** — Portugal is the only country where NUTS-3 enables a meaningful intra-country geographic split for hold-out evaluation. The top Portuguese region `PT11E` (Greater Porto area) holds ~24 % of Portuguese parcels.

### 4.5 Time-series length

![Timestep distribution per country](figures/07_timestep_hist.png)

| Country | mean | median | min | max |
|---|---:|---:|---:|---:|
| Estonia  | 46.6 | 44  | 10 | 146 |
| Latvia   | 45.7 | 45  | 10 |  82 |
| Portugal | 60.3 | 49  |  1 | 146 |

**Findings.**
- Estonia and Latvia look almost identical (mean ~46, narrow std ~7–10). Same latitude, same cloud regime.
- Portugal is **bimodal**: most parcels cluster around 40–50 like the Baltics, but a long tail extends up to ~100. That tail is the Atlantic / southern parcels — fewer winter clouds → more cloud-free S2 acquisitions retained.
- Portugal also has the worst floor (`min = 1`) — some southern parcels survived cloud filtering with almost no observations. Those need a minimum-length filter before they hit a model.

**Implication.** Variable-length series → padding + masking for any sequence model (TerraMind, THOR, attention-based ViT). The 10–146 range means **mask-aware encoders** are non-negotiable. Embedding-based pipelines (TESSERA / AlphaEarth) need a fixed-length resampling step before SVM/RF probing.

### 4.6 Spatial coverage

![Parcel centroids](figures/08_centroids.png)

Centroids land where they should (sanity check passes): Estonia + Latvia form a single contiguous Baltic mass at ~57–60 °N / 22–28 °E; Portugal is the separate Iberian cluster at ~37–42 °N / -10 to -7 °E. **No overlap, ~20 ° latitudinal gap** — climatically these are genuinely different regimes, which is the whole point of using all three for transfer experiments.

## 5. Part 2 — per-place time-series plots

The script picks the **top-3 global classes** and plots one random parcel per country: left panel = 13 raw S2 bands, right panel = NDVI.

### 5.1 HCAT 3302000000 — Grassland (present in all three countries)

![Grassland time-series](figures/09_ts_class_3302000000.png)

**What to look at.**
- **Estonia / Latvia** show the textbook northern-temperate phenology: NDVI ~0.3 in winter, single broad peak ~0.75 in June–August, sharp drop to ~0.3 by November. Sparse observations Nov–Feb (snow + clouds).
- **Portugal** is qualitatively different: NDVI is **already 0.5 in February**, plateaus around 0.5–0.7 from spring to autumn, and never drops to Baltic winter levels — Mediterranean grasslands stay green most of the year.

**Takeaway for the FM benchmark.** Same HCAT class, completely different signal. A model trained on Baltic grasslands and naively tested on Portuguese grasslands will fail unless the model learned to encode **phenological shape, not absolute timing**. This is exactly the kind of behaviour where XAI on FM embeddings becomes interesting — does TESSERA / Prithvi anchor on "July peak" (location-specific) or on "single peak per year" (shape-invariant)?

### 5.2 HCAT 3301010101 — Spring soft wheat (Estonia + Latvia only)

![Spring wheat time-series](figures/09_ts_class_3301010101.png)

Portugal panel is empty — class absent. NDVI shows the **annual-crop signature**: dormant winter (0.1–0.2), sharp rise from May, single peak ~0.75 in mid-July, rapid senescence to bare-soil values by late August.

### 5.3 Intra-class variance — three random Estonian grassland parcels

![Intra-class variance](figures/10_intraclass_variance.png)

Same country, same class, three random parcels — and the NDVI shapes already vary visibly: parcel 1 has a clean June–August peak with a small late-September bump (likely a second cut); parcel 2 plateaus high through summer; parcel 3 looks more erratic, possibly a less-intensively-managed field. **This intra-class variance is exactly the noise floor any classifier must beat** — and it argues for evaluating with a proper test set that holds out whole parcels, not random observations.

## 6. Estonia deep-dive — phenology of the top-5 classes

Aggregating across **all 175,906 Estonia parcels** (script: [_estonia_phenology.py](_estonia_phenology.py)), sampling 400 parcels per class, resampling NDVI to weekly bins, and plotting mean ± 1σ band per class.

![Estonia phenology and observation density](figures/11_estonia_phenology.png)

**Left — class phenology.**
- All five Estonian top crops share the same **single-peak summer shape** (this is just what northern-temperate agriculture looks like), but the peaks differ in timing and amplitude:
  - **3302000000 (grassland, n=400)**: gentle rise, peak ~0.75 in late June, slow decline. Widest variance band — heterogeneous management (grazing intensity, cutting frequency).
  - **3301010101 (spring wheat, n=400)**: latest peak (~mid July) and the highest amplitude (~0.85). Tight σ — well-managed monoculture.
  - **3301090300 (n=400)** (fallow / set-aside): lowest amplitude peak, narrowest dynamic range.
  - **3301010402 (winter barley)**: earliest peak (June), reflecting the autumn-sown phenology.
  - **3301010500 (oats)**: between spring wheat and grassland in timing.
- The σ-bands **overlap significantly mid-summer (~0.75 NDVI)** but **separate clearly at the shoulders** (April–May green-up; September senescence). A classifier that has access to the full temporal series should beat one that sees only peak-season — this is testable with the embedding-probe experiment.

**Right — observation density.**
- Cloud-filtering leaves **almost zero usable observations in December–February** (snow + cloud).
- Peak data density is **May–September**, with the modal week around mid-July (~7 000 observations/week across 3 000 sampled parcels).
- This is a hard constraint for AlphaEarth: its embeddings are *annual* and effectively averaged over the whole calendar year. With Estonia having most of its signal compressed into May–Sep and ~zero data in winter, AlphaEarth's annual average should look closer to a summer composite than to a true annual statistic. We should test this empirically by comparing AlphaEarth vs TESSERA embeddings on the same parcels.

## 7. Headline conclusions

1. **Scale is honest.** 706 683 .npz files match the published count exactly. Format is simple (3 numpy arrays per file) and trivially loadable — no hidden gotchas.
2. **The dataset is extremely imbalanced.** Grassland is 45 %; top-10 classes cover 80 %; the long tail (~150 classes) is barely usable for supervised learning. Macro-averaged metrics + class-restricted protocols are mandatory.
3. **Transnational transfer is genuinely cross-distribution.** Only 11 of the top-20 classes appear in all three countries, and the same class shows **qualitatively different phenology** in Portugal vs the Baltics (grassland NDVI plot). This is the right benchmark for testing whether a foundation model has learned location-invariant agricultural representations or just memorised regional priors.
4. **Variable-length, summer-skewed time series.** Mean 46 obs/parcel in the Baltics, 60 in Portugal, but with hard zeros in winter for the Baltics. Embedding-probe pipelines need a fixed-length resampler; sequence models need mask-aware attention.
5. **AlphaEarth caveat is real.** Estonia's data density collapses Nov–Feb, so AlphaEarth's annual embedding will effectively summarise a summer composite. Expect AlphaEarth to underperform TESSERA / TerraMind on Baltic crops where the diagnostic information sits in shoulder-season dynamics — and report this as a *finding*, not a bug.
6. **Portugal is the hardest, most informative target.** Smallest sample size (~100 k parcels), most NUTS-3 regions for intra-country splits, mostly disjoint label space from EE/LV. Few-shot results on Portugal will be the thesis's most defensible cross-distribution result.

### Direct next steps

- Pull the EuroCrops HCAT taxonomy CSV (from [maja601/EuroCrops](https://github.com/maja601/EuroCrops)) and join on `hcat` to get human-readable crop names — replace HCAT ids in every figure.
- Use `eurocropsml-cli datasets eurocrops build-splits` to generate the official k-shot + transnational splits before any classifier training.
- For the embedding-probe experiment: sample TESSERA / AlphaEarth at the `center` coordinates stored in each .npz, join on `parcel_id`, and run SVM/RF probes on Estonia's top-5 classes first (the deep-dive set above) — that's a small, well-understood problem that will quickly tell us whether the FM embeddings carry the phenological information the raw bands obviously do.